# Notebook 05 — Foundation Model (Chronos via TimeCoPilot)

**ISA 444 Final Project — Retail Forecasting (Walmart)**

Mirrors the structure of `duke_energy_transformers.ipynb` from class.

### What this notebook does
1. Use Amazon's **Chronos** time-series foundation model in **zero-shot** mode — no training, just inference.
2. Run 5-fold non-overlapping CV at `h = 4`.
3. Evaluate ME / MAE / RMSE / MAPE per series.
4. Produce future forecasts for the Kaggle test dates.

### What is a foundation model and why does it matter?
Chronos is pre-trained on a massive corpus of time-series data from many domains. It learns 'how time series in general behave' once, then any new series can be forecast **without fitting anything** — you just hand it the history and it returns predictions. This is the same paradigm as LLMs (pre-train once, use everywhere) applied to forecasting.

Pros:
- **No training** = instant deployment, no hyperparameters to tune.
- **Often competitive** with hand-tuned classical models out of the box.
- Performs well on **short series** where deep models struggle (we only have 143 weeks per series).

Cons:
- **No exog support** — Chronos is univariate. It can't see our holiday/markdown flags.
- **Bigger memory footprint** than a tree model — but `chronos-bolt-small` is small enough for 8 GB.
- **First use downloads ~200 MB** of weights from HuggingFace; cached after that.

### Why `chronos-bolt-small`?
`bolt-tiny` is faster but accuracy drops. `bolt-base` is more accurate but ~4x bigger. `bolt-small` is the sweet spot for an 8 GB laptop and is what the TimeCoPilot quickstart examples use.


## Install Packages

Run once on a fresh environment. `timecopilot` pulls in `transformers` and the Chronos repo automatically.

**Note:** TimeCoPilot has had Colab/pandas conflicts in the past. On Windows + a fresh venv this should install cleanly, but if you hit an error, run the install cell twice (the second pass usually resolves any dependency reshuffles).

In [2]:
#!pip install timecopilot utilsforecast pyarrow
#!pip install time-copilot utilsforecast pyarrow chronos-forecasting
#!pip install timecopilot


ERROR: Could not find a version that satisfies the requirement time-copilot (from versions: none)
ERROR: No matching distribution found for time-copilot


  Using cached timecopilot-0.0.25-py3-none-any.whl.metadata (17 kB)
  Using cached arch-8.0.0-cp311-cp311-win_amd64.whl.metadata (13 kB)
  Using cached datasets-4.8.5-py3-none-any.whl.metadata (19 kB)
  Using cached lightning-2.4.0-py3-none-any.whl.metadata (38 kB)
  Using cached logfire-4.11.0-py3-none-any.whl.metadata (10 kB)
  Using cached nixtla-0.7.4-py3-none-any.whl.metadata (15 kB)
  Using cached opentelemetry_instrumentation-0.62b1-py3-none-any.whl.metadata (7.2 kB)
  Using cached opentelemetry_sdk-1.37.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached peft-0.19.1-py3-none-any.whl.metadata (15 kB)
  Using cached prophet-1.3.0-py3-none-win_amd64.whl.metadata (3.6 kB)
  Using cached pydantic_ai-1.96.0-py3-none-any.whl.metadata (15 kB)
  Using cached tabpfn_time_series-1.0.3-py3-none-any.whl.metadata (4.9 kB)
  Using cached timecopilot_chronos_forecasting-0.2.1-py3-none-any.whl.metadata (24 kB)
  Using cached timecopilot_granite_tsfm-0.1.3-py3-none-any.whl.metadata (18 kB)
  Usi

ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: 'C:\\Users\\23mwa\\AppData\\Local\\Packages\\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\\LocalCache\\local-packages\\Python311\\site-packages\\prophet\\stan_model\\cmdstan-2.37.0\\stan\\lib\\stan_math\\lib\\tbb_2020.3\\include\\tbb\\internal\\_deprecated_header_message_guard.h'
HINT: This error might have occurred since this system does not have Windows Long Path support enabled. You can find information on how to enable this at https://pip.pypa.io/warnings/enable-long-paths



## Library Imports

In [2]:
import sys

!{sys.executable} -m pip install tsfeatures

  Using cached tsfeatures-0.4.5-py3-none-any.whl.metadata (7.4 kB)
  Using cached arch-8.0.0-cp311-cp311-win_amd64.whl.metadata (13 kB)
Using cached tsfeatures-0.4.5-py3-none-any.whl (28 kB)
Using cached arch-8.0.0-cp311-cp311-win_amd64.whl (937 kB)

   ---------------------------------------- 0/2 [arch]
   ---------------------------------------- 0/2 [arch]
   ---------------------------------------- 0/2 [arch]
   ---------------------------------------- 0/2 [arch]
   ---------------------------------------- 0/2 [arch]
   -------------------- ------------------- 1/2 [tsfeatures]
   ---------------------------------------- 2/2 [tsfeatures]



ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
timecopilot 0.0.25 requires datasets>=4.1.1, which is not installed.
timecopilot 0.0.25 requires lightning==2.4.0, which is not installed.
timecopilot 0.0.25 requires logfire==4.11.0, which is not installed.
timecopilot 0.0.25 requires nixtla>=0.7.0, which is not installed.
timecopilot 0.0.25 requires opentelemetry-instrumentation>=0.58b0, which is not installed.
timecopilot 0.0.25 requires opentelemetry-sdk==1.37.0, which is not installed.
timecopilot 0.0.25 requires peft<1,>=0.13.0, which is not installed.
timecopilot 0.0.25 requires prophet>=1.1.7, which is not installed.
timecopilot 0.0.25 requires pydantic-ai>=0.7.0, which is not installed.
timecopilot 0.0.25 requires tabpfn-time-series==1.0.3; python_full_version < "3.13", which is not installed.
timecopilot 0.0.25 requires timecopilot-chronos-forecasting>=0

In [3]:
import sys

!{sys.executable} -m pip uninstall -y prophet timecopilot
!{sys.executable} -m pip install timecopilot --no-deps
!{sys.executable} -m pip install utilsforecast pyarrow chronos-forecasting transformers accelerate torch pandas numpy

  Using cached timecopilot-0.0.25-py3-none-any.whl.metadata (17 kB)
Using cached timecopilot-0.0.25-py3-none-any.whl (102 kB)


In [3]:
import os
import warnings
import pandas as pd
import numpy as np
from pathlib import Path

from timecopilot import TimeCopilotForecaster
from timecopilot.models.foundation.chronos import Chronos

from utilsforecast.evaluation import evaluate
from utilsforecast.losses import bias, mae, rmse, mape

warnings.filterwarnings("ignore")
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("Imports worked")

 See https://github.com/google-research/timesfm/blob/master/README.md for updated APIs.
Imports worked


## Paths, Settings, and Cache Toggles

Same caching approach as Notebook 04 — even though Chronos is fast, we cache so that re-running this notebook to inspect outputs doesn't re-trigger inference.

In [4]:
DATA_DIR = Path("../data")
OUT_DIR  = Path("../outputs")
(OUT_DIR / "cv_results").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "forecasts").mkdir(parents=True, exist_ok=True)

FREQ      = "W-FRI"
H         = 4
N_WINDOWS = 5
STEP_SIZE = 4

# Chronos model — bolt-small is the sweet spot for an 8 GB laptop.
CHRONOS_REPO = "amazon/chronos-bolt-small"

# Cache toggles. Set True to force a re-run.
FORCE_CV_RERUN     = False
FORCE_FUTURE_RERUN = False

CV_CACHE_PATH     = OUT_DIR / "cv_results" / "cv_chronos_predictions.csv"
FUTURE_CACHE_PATH = OUT_DIR / "forecasts"  / "test_chronos.csv"

print("Chronos model:", CHRONOS_REPO)

Chronos model: amazon/chronos-bolt-small


## Load Training Data

Univariate only — Chronos doesn't take exog.

In [5]:
df_train = pd.read_parquet(DATA_DIR / "walmart_top20_train.parquet")[["unique_id", "ds", "y"]]
df_test  = pd.read_parquet(DATA_DIR / "walmart_top20_test.parquet")[["unique_id", "ds"]]

print("Training rows  :", len(df_train))
print("Training series:", df_train["unique_id"].nunique())
df_train.head()

Training rows  : 2860
Training series: 20


,unique_id,ds,y
0,S10_D72,2010-02-05,232558.51
1,S10_D72,2010-02-12,202622.42
2,S10_D72,2010-02-19,184982.01
3,S10_D72,2010-02-26,186002.43
4,S10_D72,2010-03-05,191989.54


## Configure The Forecaster

`TimeCopilotForecaster` wraps any foundation model with the standard Nixtla `cross_validation` / `forecast` API, so the output schema matches what we got from `statsforecast` and `mlforecast` — drop-in compatible with our evaluation step.


In [6]:
# Build the Chronos wrapper. alias='Chronos' so the prediction column ends up
# named the same way every other notebook names its model column.
chronos_model = Chronos(repo_id=CHRONOS_REPO, alias="Chronos")

tcf = TimeCopilotForecaster(models=[chronos_model])
print("Forecaster ready. First inference call will download ~200 MB of weights if not cached.")

Forecaster ready. First inference call will download ~200 MB of weights if not cached.


## Cross-Validation (Cached)

Even though Chronos has no training step, we still run CV with `n_windows=5, step_size=4` so we can compare its 5-fold metrics against the other models. At each fold:
- Chronos receives the history up to the fold cutoff.
- Predicts the next 4 weeks.
- Predictions are compared to actuals.

Expected runtime: **5-15 minutes** on first run (mostly the one-time model download). After that, much faster.


In [7]:
if CV_CACHE_PATH.exists() and not FORCE_CV_RERUN:
    print("[CACHED] Loading existing CV predictions from", CV_CACHE_PATH.resolve())
    cv_chronos = pd.read_csv(CV_CACHE_PATH, parse_dates=["ds", "cutoff"])
    print("          Shape:", cv_chronos.shape)
else:
    print("[RUNNING] Cross-validating Chronos — first call downloads ~200 MB of weights.")
    cv_chronos = tcf.cross_validation(
        df         = df_train,
        h          = H,
        freq       = FREQ,
        n_windows  = N_WINDOWS,
        step_size  = STEP_SIZE,
    )
    cv_chronos.to_csv(CV_CACHE_PATH, index=False)
    print("[SAVED]   CV predictions ->", CV_CACHE_PATH.resolve())
    print("          Shape:", cv_chronos.shape)

cv_chronos.head()

[RUNNING] Cross-validating Chronos — first call downloads ~200 MB of weights.


0it [00:00, ?it/s]

config.json: 0.00B [00:00, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/191M [00:00<?, ?B/s]

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.
100%|██████████| 2/2 [00:00<00:00,  4.10it/s]
5it [00:27,  5.41s/it]

[SAVED]   CV predictions -> C:\Users\23mwa\outputs\cv_results\cv_chronos_predictions.csv
          Shape: (400, 5)


,unique_id,ds,cutoff,y,Chronos
0,S10_D72,2012-06-15,2012-06-08,105499.39,114454.718750
1,S10_D72,2012-06-22,2012-06-08,107949.41,117954.593750
2,S10_D72,2012-06-29,2012-06-08,96579.10,119750.828125
3,S10_D72,2012-07-06,2012-06-08,100464.25,122198.828125
4,S13_D90,2012-06-15,2012-06-08,122102.95,120812.890625


## Evaluate Per Series, Per Metric

In [8]:
eval_chronos = evaluate(
    cv_chronos,
    metrics = [bias, mae, rmse, mape],
    models  = ["Chronos"],
)

eval_chronos.to_csv(OUT_DIR / "cv_results" / "eval_chronos.csv", index=False)
print("Saved:", (OUT_DIR / "cv_results" / "eval_chronos.csv").resolve())
eval_chronos.head(8)

Saved: C:\Users\23mwa\outputs\cv_results\eval_chronos.csv


,unique_id,cutoff,metric,Chronos
0,S10_D72,2012-06-08,bias,15966.704687
1,S13_D90,2012-06-08,bias,1309.230312
2,S13_D92,2012-06-08,bias,-1490.727031
3,S13_D95,2012-06-08,bias,-331.705625
4,S14_D92,2012-06-08,bias,29533.140312
5,S14_D95,2012-06-08,bias,21106.633594
6,S19_D92,2012-06-08,bias,872.175234
7,S1_D92,2012-06-08,bias,-3736.693828


In [9]:
# Aggregate across all 20 series.
agg_chronos = (
    eval_chronos
    .drop(columns=["unique_id"])
    .groupby("metric")
    .mean()
    .round(2)
)
display(agg_chronos)
agg_chronos.to_csv(OUT_DIR / "cv_results" / "eval_chronos_aggregate.csv")

,cutoff,Chronos
metric,,
bias,2012-08-03,2359.35
mae,2012-08-03,6267.77
mape,2012-08-03,0.05
rmse,2012-08-03,7274.67


## Future Forecasts For The Kaggle Test Dates (Cached)

TimeCopilot's `forecast()` API takes the training history and the horizon `h`, and returns predictions for the next `h` weeks. Since the Kaggle test horizon is 39 weeks, we call `forecast(h=39)`.

Unlike NeuralForecast, foundation models can predict arbitrary horizons in one shot — no rolling needed.


In [10]:
h_future = df_test.groupby("unique_id").size().max()
print("Future horizon (weeks):", h_future)

if FUTURE_CACHE_PATH.exists() and not FORCE_FUTURE_RERUN:
    print("[CACHED] Loading future forecasts from", FUTURE_CACHE_PATH.resolve())
    fc_chronos = pd.read_csv(FUTURE_CACHE_PATH, parse_dates=["ds"])
else:
    print("[RUNNING] Producing future forecasts (39 weeks)...")
    fc_chronos = tcf.forecast(
        df   = df_train,
        h    = h_future,
        freq = FREQ,
    )
    fc_chronos.to_csv(FUTURE_CACHE_PATH, index=False)
    print("[SAVED]   Future forecasts ->", FUTURE_CACHE_PATH.resolve())

print("Shape:", fc_chronos.shape)
fc_chronos.head()

Future horizon (weeks): 39
[RUNNING] Producing future forecasts (39 weeks)...


100%|██████████| 2/2 [00:00<00:00,  2.67it/s]

[SAVED]   Future forecasts -> C:\Users\23mwa\outputs\forecasts\test_chronos.csv
Shape: (780, 3)


,unique_id,ds,Chronos
0,S10_D72,2012-11-02,122741.890625
1,S10_D72,2012-11-09,129646.593750
2,S10_D72,2012-11-16,136933.718750
3,S10_D72,2012-11-23,142980.078125
4,S10_D72,2012-11-30,143283.171875


## Notebook 05 — Done

**Outputs:**
- `..\outputs\cv_results\cv_chronos_predictions.csv` — every CV fold prediction
- `..\outputs\cv_results\eval_chronos.csv` — per-series, per-metric evaluation
- `..\outputs\cv_results\eval_chronos_aggregate.csv` — mean across series
- `..\outputs\forecasts\test_chronos.csv` — future forecasts on Kaggle test dates

**Re-run behavior:**
- Default → uses cached CSVs (instant).
- `FORCE_CV_RERUN=True` → re-runs CV (5-15 min).
- `FORCE_FUTURE_RERUN=True` → re-runs future forecasts (1-2 min).

**Next:** Notebook 06 — the synthesis notebook. Combines all model evaluation tables, counts wins per metric, generates forecast-vs-actual plots for every series, and writes the final summary.
